# Install Dependencies

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model

In [ ]:
import torch
print(torch.cuda.is_available())

True


# Import Libraries

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

# Load GSM8K Dataset

In [ ]:
dataset = load_dataset("openai/gsm8k", "main")

train_data = dataset["train"].select(range(3000))
test_data = dataset["test"].select(range(1000))

print(train_data[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


# Load Model

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

# Format Dataset

In [ ]:
def format_prompt(example):
    return {
        "text": f"Question: {example['question']}\nAnswer: {example['answer']}"
    }

train_data = train_data.map(format_prompt)
test_data = test_data.map(format_prompt)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

# Tokenization

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

# Apply LoRA

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    logging_steps=50,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    learning_rate=2e-4,
    fp16=True,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data
)

In [ ]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
def tokenize(example):
    model_inputs = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

train_data = train_data.map(tokenize, batched=True, remove_columns=train_data.column_names)
test_data = test_data.map(tokenize, batched=True, remove_columns=test_data.column_names)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
train_data.set_format(type="torch")
test_data.set_format(type="torch")

# Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data
)

# Train Model

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
200,0.376457,0.395558
400,0.352309,0.395691
600,0.369250,0.393941
800,0.392201,0.392103
1000,0.378482,0.390435
1200,0.415410,0.388518
1400,0.406512,0.387126
1600,0.321918,0.387158
1800,0.353689,0.384902
2000,0.343154,0.384167


Step,Training Loss,Validation Loss
200,0.376457,0.395558
400,0.352309,0.395691
600,0.369250,0.393941
800,0.392201,0.392103
1000,0.378482,0.390435
1200,0.415410,0.388518
1400,0.406512,0.387126
1600,0.321918,0.387158
1800,0.353689,0.384902
2000,0.343154,0.384167


TrainOutput(global_step=3000, training_loss=0.36835059547424315, metrics={'train_runtime': 1797.3133, 'train_samples_per_second': 1.669, 'train_steps_per_second': 1.669, 'total_flos': 9576261856788480.0, 'train_loss': 0.36835059547424315, 'epoch': 1.0})

## **Evaluate Model**

In [ ]:
results = trainer.evaluate()
print(results)

Step,Training Loss,Validation Loss
200,0.424126,0.433723
400,0.391008,0.420371
600,0.403457,0.413236
800,0.419709,0.408577
1000,0.403741,0.404898
1200,0.438585,0.401938
1400,0.430439,0.399034
1600,0.339241,0.398497
1800,0.369912,0.395621
2000,0.358748,0.394401


Step,Training Loss,Validation Loss
200,0.424126,0.433723
400,0.391008,0.420371
600,0.403457,0.413236
800,0.419709,0.408577
1000,0.403741,0.404898
1200,0.438585,0.401938
1400,0.430439,0.399034
1600,0.339241,0.398497
1800,0.369912,0.395621
2000,0.358748,0.394401


{'eval_loss': 0.3924177289009094}


# **Test Model**

In [ ]:
def test_model(question):
    input_text = f"Question: {question}\nAnswer:"
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=100
    )

    print(tokenizer.decode(output[0]))

test_model("If John has 5 apples and buys 3 more, how many apples?")

<s> Question: If John has 5 apples and buys 3 more, how many apples?
Answer: If John has 5 apples, he has 5+3=<<5+3=8>>8 apples.
If he buys 3 more apples, he has 8+3=<<8+3=11>>11 apples.
#### 11</s>


In [2]:
!pip install gradio

In [ ]:
import gradio as gr
import torch

def predict(question):
    input_text = f"Question: {question}\nAnswer:"
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=100
    )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response

interface = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=2, placeholder="Ask your question here..."),
    outputs="text",
    title="GSM8K Reasoning Model",
    description="Ask math reasoning questions"
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9abcfe619adb5d80f4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
